# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library. 

### Dataset Source
The dataset is defined via a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant library (uncomment the next line if running in a clean environment)
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)

## 2. Data Overview
List available record sets and their field `@id`s. All identifiers are referenced by their unique `@id`.

In [ ]:
# Display all available record sets and their fields using @id

record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found directly in the metadata. Looking for record sets in distributions...")
    # Try getting from distributions if record sets are declared there
    distributions = getattr(metadata, 'distribution', [])
    print(f"Distributions found ({len(distributions)}):")
    for dist in distributions:
        print("  -", getattr(dist, '@id', str(dist)))
        # Attempt to load record sets for each distribution
        try:
            ds2 = mlc.Dataset(getattr(dist, '@id', dist))
            for rs in ds2.record_sets:
                print(f"  RecordSet: {rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', 'n/a')}")
                fields = getattr(rs, 'fields', []) or rs.get('field', []) if isinstance(rs, dict) else []
                for field in fields:
                    print(f"    Field: {field['@id'] if isinstance(field, dict) else getattr(field, '@id', 'n/a')}")
        except Exception as e:
            print(f"    Could not read: {e}")
else:
    print(f"Record Sets found ({len(record_sets)}):")
    for rs in record_sets:
        print(f"- RecordSet @id: {getattr(rs, '@id', 'n/a')}")
        fields = getattr(rs, 'fields', []) or getattr(rs, 'field', [])
        for field in fields:
            print(f"    Field @id: {getattr(field, '@id', 'n/a')}")

## 3. Data Extraction
Extract the dataset records for further exploration.

**Note:** In some datasets, records are structured under different record sets or distributions. If only one record set is available, we use its `@id`.

In [ ]:
# Find all available record set @id's
record_set_ids = [getattr(rs, '@id', None) for rs in dataset.record_sets]
record_set_ids = [x for x in record_set_ids if x is not None]

dataframes = {}
if not record_set_ids:
    print("No record sets in the main metadata. Checking distributions for record sets...")
    # Try loading directly from each distribution (file)
    from collections.abc import Iterable
    distributions = getattr(metadata, 'distribution', [])
    for dist in distributions:
        dist_id = getattr(dist, '@id', str(dist))
        try:
            ds_dist = mlc.Dataset(dist_id)
            record_sets_dist = list(ds_dist.record_sets)
            for rs in record_sets_dist:
                rs_id = getattr(rs, '@id', None)
                if rs_id:
                    print(f"Loading records for RecordSet {rs_id}")
                    records = list(ds_dist.records(record_set=rs_id))
                    df = pd.DataFrame(records)
                    dataframes[rs_id] = df
        except Exception as e:
            print(f"Error loading distribution {dist_id}: {e}")
else:
    print(f"Loading records for record sets: {record_set_ids}")
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

# Print record set IDs and the first few columns of one DataFrame
if not dataframes:
    print("No records/dataframes loaded.")
else:
    for rs_id in dataframes:
        print(f"RecordSet @id: {rs_id}")
        print("Columns:", dataframes[rs_id].columns.tolist())
        display(dataframes[rs_id].head())
        break  # Display only the first loaded DataFrame

## 4. Exploratory Data Analysis (EDA)
Perform common data processing steps using appropriate field `@id`s. We'll select a numeric field, apply filtering, normalization, and try grouping by a categorical field.

*You may need to update the field `@id`s below to match those present in the actual dataset columns (check outputs above!)*

In [ ]:
# Choose the first DataFrame and display column @id's for reference
if not dataframes:
    print("No dataframes available for EDA.")
else:
    # Pick first loaded record set
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"Available columns (@id): {df.columns.tolist()}")
    # --- Replace the following field @id's with those present in your data! ---
    # Suppose common @id's for numeric field and group field
    possible_numeric = [col for col in df.columns if df[col].dtype != object][:1]
    if possible_numeric:
        numeric_field = possible_numeric[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        numeric_field = df.columns[0]
        print("No obvious numeric column, using first column as numeric_field:", numeric_field)

    # For grouping, try to pick an object-type column
    possible_group = [col for col in df.columns if df[col].dtype == object]
    group_field = possible_group[0] if possible_group else df.columns[0]

    # Filter records where numeric_field > threshold
    threshold = 10
    try:
        filtered_df = df[df[numeric_field] > threshold].copy()
    except Exception as e:
        print(f"Filtering with >{threshold} failed: {e}\nTrying conversion to numeric...")
        filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()

    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the chosen numeric field
    filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') -
                                                  pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()) / \
                                                pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by the group_field and aggregate means
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (mean):")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalization, and plot group means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce').dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")

    plt.subplot(1,2,2)
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=20)
    plt.title(f"Normalized {numeric_field}")
    plt.tight_layout()
    plt.show()

    # If we have a grouped mean, plot it as bar
    if 'grouped_df' in locals() and not grouped_df.empty and numeric_field in grouped_df.columns:
        grouped_df[numeric_field].plot(kind='bar', figsize=(10,5), title=f"Group means of {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process the **Ordered Logistic Regression Results for Adoption Predictors** dataset using the `mlcroissant` library. Key steps included loading the Croissant schema, listing record sets and fields by their unique `@id`, extracting records into pandas DataFrames, performing standard data filtering and normalization, and visualizing numeric distributions. This workflow is adaptable for any Croissant-compatible dataset with record set and field identifiers.